In [1]:
import torch
import cv2
import math 
import numpy as np
import mediapipe as mp
from ultralytics import YOLO
import torch




In [2]:


# 1. 3D Model Points of standard facial features (in arbitrary world coordinates)
# Nose tip, Chin, Left Eye corner, Right Eye corner, Left Mouth corner, Right Mouth corner
model_points = np.array([
    (0.0, 0.0, 0.0),             # Nose tip
    (0.0, -330.0, -65.0),        # Chin
    (-225.0, 170.0, -135.0),     # Left eye corner
    (225.0, 170.0, -135.0),      # Right eye corner
    (-150.0, -150.0, -125.0),    # Left mouth corner
    (150.0, -150.0, -125.0)      # Right mouth corner
], dtype=np.float32)

# 2. Specific MediaPipe landmark indices that match the 3D model points above
TRACKED_FACE_INDICES = [1, 152, 33, 263, 61, 291]

# Initialize the new state-machine timer for head turns
head_turn_frames = 0

In [3]:
def get_distance(p1,p2):
    return math.hypot(p2[0]-p1[0],p1[1]-p2[1])

def calculate_ear(eye_indices,landmarks,frame_width,frame_height):
    points=[]
    for idx in eye_indices:
        x=int(landmarks.landmark[idx].x*frame_width)
        y=int(landmarks.landmark[idx].y*frame_height)
        points.append((x,y))

    v1=get_distance(points[1],points[5])
    v2=get_distance(points[2],points[4])

    h=get_distance(points[0],points[3])

    ear=(v1+v2)/(2.0*h)
    return ear,points

# Media Pipe Set up 

mp_face_mesh=mp.solutions.face_mesh
face_mesh=mp_face_mesh.FaceMesh(max_num_faces=1,refine_landmarks=True)

LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

In [4]:


# Load the Edge-Optimized YOLO model and push to your RTX 4060
yolo_model = YOLO('yolov8n.pt')
if torch.cuda.is_available():
    yolo_model.to('cuda')

# Initialize Phase 7 Memory Variables
frame_counter = 0
current_yolo_boxes = [] # This remembers the bounding boxes during the "skip frames"

In [5]:
# ==========================================================
# CELL 3: THE EXECUTION ENGINE
# ==========================================================

cap = cv2.VideoCapture(0)

# Initialize Core Variables
risk_score = 0
frame_counter = 0

# Initialize State Machine Timers
eyes_closed_frames = 0
head_turn_frames = 0
no_faces_frames = 0
multi_faces_frames = 0
current_yolo_boxes = []

while True:
    success, frame = cap.read()
    if not success:
        break

    h, w, _ = frame.shape
    frame_counter += 1

    # --- THE KILL SWITCH OVERRIDE ---
    if risk_score >= 100:
        cv2.rectangle(frame, (0, 0), (w, h), (0, 0, 255), -1) # Fills screen with Red
        cv2.putText(frame, "EXAM TERMINATED", (w//2 - 200, h//2), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 3)
        cv2.putText(frame, "Contact Administrator", (w//2 - 150, h//2 + 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.imshow('Ultimate AI Proctoring System', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        continue # Freezes all other processing


    # ==========================================================
    # PHASE 7: SIMPLIFIED YOLO PROCESSING 
    # ==========================================================
    if frame_counter % 15 == 0:
        current_yolo_boxes = []
        results_yolo = yolo_model(frame, verbose=False)
        
        for r in results_yolo:
            for box in r.boxes:
                cls_id = int(box.cls[0])
                # Direct Penalty Logic: If seen on this 15th frame, add penalty immediately
                if cls_id == 67: # Phone
                    x1, y1, x2, y2 = box.xyxy[0]
                    current_yolo_boxes.append((int(x1), int(y1), int(x2), int(y2), "Phone"))
                    risk_score = min(100, risk_score + 10) 
                elif cls_id == 73: # Book
                    x1, y1, x2, y2 = box.xyxy[0]
                    current_yolo_boxes.append((int(x1), int(y1), int(x2), int(y2), "Book"))
                    risk_score = min(100, risk_score + 10)

    # Draw persistent YOLO boxes on the video canvas
    for (x1, y1, x2, y2, label) in current_yolo_boxes:
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(frame, f"UNAUTHORIZED: {label}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    # ==========================================================
    # MEDIAPIPE TELEMETRY & MATH
    # ==========================================================
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb_frame)

    # Camera math for 3D mapping
    focal_length = w
    center = (w / 2, h / 2)
    camera_matrix = np.array([
        [focal_length, 0, center[0]],
        [0, focal_length, center[1]],
        [0, 0, 1]
    ], dtype=np.float32)
    dist_coeffs = np.zeros((4, 1))

    # Clean UI Telemetry Dashboard Layout
    cv2.putText(frame, f"SYSTEM MONITOR", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    risk_color = (0, 0, 255) if risk_score > 50 else (0, 255, 0)
    cv2.putText(frame, f"Risk Level: {risk_score}/100", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.6, risk_color, 2)

    # Face Tracking Logic
    if results.multi_face_landmarks:
        no_faces_frames = 0 
        
        # Multiple Faces Trap
        if len(results.multi_face_landmarks) > 1:
            multi_faces_frames += 1
            if multi_faces_frames > 30:
                risk_score = min(100, risk_score + 50)
                multi_faces_frames = 0
            cv2.putText(frame, "STATUS: MULTIPLE FACES!", (20, 240), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
            
        else:
            multi_faces_frames = 0
            
            for face_landmarks in results.multi_face_landmarks:
                # 1. EAR (Blink Tracking)
                left_ear, _ = calculate_ear(LEFT_EYE, face_landmarks, w, h)
                right_ear, _ = calculate_ear(RIGHT_EYE, face_landmarks, w, h)
                avg_ear = (left_ear + right_ear) / 2.0
                
                if avg_ear < 0.20:
                    eyes_closed_frames += 1
                    if eyes_closed_frames > 60:
                        risk_score = min(100, risk_score + 10)
                        eyes_closed_frames = 0
                else:
                    eyes_closed_frames = 0
                
                # 2. 3D Angle Math (solvePnP)
                image_points = []
                for idx in TRACKED_FACE_INDICES:
                    lm = face_landmarks.landmark[idx]
                    image_points.append((int(lm.x * w), int(lm.y * h)))
                image_points = np.array(image_points, dtype=np.float32)

                _, rotation_vector, translation_vector = cv2.solvePnP(
                    model_points, image_points, camera_matrix, dist_coeffs, flags=cv2.SOLVEPNP_ITERATIVE
                )

                rotation_matrix, _ = cv2.Rodrigues(rotation_vector)
                proj_matrix = np.hstack((rotation_matrix, translation_vector))
                _, _, _, _, _, _, euler_angles = cv2.decomposeProjectionMatrix(proj_matrix)
                
                pitch = euler_angles[0][0]
                yaw = euler_angles[1][0]
                
                # Print Clean Telemetry to Sidebar
                cv2.putText(frame, f"EAR: {avg_ear:.2f}", (20, 130), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
                cv2.putText(frame, f"Yaw: {int(yaw)} deg", (20, 160), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
                cv2.putText(frame, f"Pitch: {int(pitch)} deg", (20, 190), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

                # 3. Head Angle State Machine
                if abs(yaw) > 50 or pitch < -180:
                    head_turn_frames += 1
                    cv2.putText(frame, "STATUS: HEAD ANOMALY", (20, 240), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 165, 255), 1)
                    if head_turn_frames > 90:
                        risk_score = min(100, risk_score + 20)
                        head_turn_frames = 0
                else:
                    head_turn_frames = 0
                    if eyes_closed_frames > 15:
                        cv2.putText(frame, "STATUS: EYES CLOSED", (20, 240), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 165, 255), 1)
                    else:
                        cv2.putText(frame, "STATUS: NOMINAL", (20, 240), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    # ==========================================================
    # MISSING FACE LOGIC
    # ==========================================================
    else:
        no_faces_frames += 1
        if no_faces_frames > 30:
            risk_score = min(100, risk_score + 25)
            no_faces_frames = 0
        cv2.putText(frame, "STATUS: NO FACE DETECTED", (20, 240), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

    # Render final frame
    cv2.imshow('Ultimate AI Proctoring System', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

C:\Users\Admin\AppData\Roaming\Python\Python310\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


In [9]:
import cv2
import math
import numpy as np
import torch
from ultralytics import YOLO
import mediapipe as mp

# ==========================================================
# 1. MATH & HELPERS
# ==========================================================
def get_distance(p1, p2):
    return math.hypot(p2[0] - p1[0], p2[1] - p1[1])

def calculate_ear(eye_indices, landmarks, frame_width, frame_height):
    points = [(int(landmarks.landmark[idx].x * frame_width), int(landmarks.landmark[idx].y * frame_height)) for idx in eye_indices]
    hor_dist = get_distance(points[0], points[3])
    ver_dist1 = get_distance(points[1], points[5])
    ver_dist2 = get_distance(points[2], points[4])
    if hor_dist == 0: return 0.0, points
    return (ver_dist1 + ver_dist2) / (2.0 * hor_dist), points

def draw_hud_text(img, text, position, font_scale, color):
    x, y = position
    # Thick black outline with Anti-Aliasing for readability on any background
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0), 5, cv2.LINE_AA) 
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, font_scale, color, 2, cv2.LINE_AA)

LEFT_EYE = [362, 385, 387, 263, 373, 380]
RIGHT_EYE = [33, 160, 158, 133, 153, 144]
TRACKED_FACE_INDICES = [1, 152, 33, 263, 61, 291]

model_points = np.array([
    (0.0, 0.0, 0.0),             # Nose tip
    (0.0, -330.0, -65.0),        # Chin
    (-225.0, 170.0, -135.0),     # Left eye corner
    (225.0, 170.0, -135.0),      # Right eye corner
    (-150.0, -150.0, -125.0),    # Left mouth corner
    (150.0, -150.0, -125.0)      # Right mouth corner
], dtype=np.float32)

# ==========================================================
# 2. MODEL INITIALIZATION
# ==========================================================
print("Loading Models...")
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(max_num_faces=2, refine_landmarks=True)

yolo_model = YOLO('yolov8n.pt')
if torch.cuda.is_available():
    yolo_model.to('cuda')

# ==========================================================
# 3. CORE EXECUTION ENGINE
# ==========================================================
cap = cv2.VideoCapture(0)

risk_score = 0
frame_counter = 0
eyes_closed_frames = 0
head_turn_frames = 0
no_faces_frames = 0
multi_faces_frames = 0
current_yolo_boxes = []

while True:
    success, frame = cap.read()
    if not success: break
    h, w, _ = frame.shape
    frame_counter += 1

    # --- KILL SWITCH OVERRIDE ---
    if risk_score >= 100:
        cv2.rectangle(frame, (0, 0), (w, h), (0, 0, 255), -1) 
        draw_hud_text(frame, "EXAM TERMINATED", (w//2 - 200, h//2), 1.5, (255, 255, 255))
        draw_hud_text(frame, "Contact Administrator", (w//2 - 150, h//2 + 40), 0.8, (255, 255, 255))
        cv2.imshow('Ultimate AI Proctoring System', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'): break
        continue 

    # --- ASYNCHRONOUS YOLO (15-Frame Skip Logic) ---
    if frame_counter % 15 == 0:
        current_yolo_boxes = []
        results_yolo = yolo_model(frame, verbose=False)
        for r in results_yolo:
            for box in r.boxes:
                cls_id = int(box.cls[0])
                if cls_id == 67: # Phone
                    x1, y1, x2, y2 = box.xyxy[0]
                    current_yolo_boxes.append((int(x1), int(y1), int(x2), int(y2), "Phone"))
                    risk_score = min(100, risk_score + 10) 
                elif cls_id == 73: # Book
                    x1, y1, x2, y2 = box.xyxy[0]
                    current_yolo_boxes.append((int(x1), int(y1), int(x2), int(y2), "Book"))
                    risk_score = min(100, risk_score + 10)

    for (x1, y1, x2, y2, label) in current_yolo_boxes:
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
        draw_hud_text(frame, f"UNAUTHORIZED: {label}", (x1, y1 - 10), 0.6, (0, 0, 255))

    # --- MEDIAPIPE & SPATIAL GEOMETRY ---
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb_frame)

    camera_matrix = np.array([[w, 0, w / 2], [0, w, h / 2], [0, 0, 1]], dtype=np.float32)
    dist_coeffs = np.zeros((4, 1))

    # General HUD Readouts
    draw_hud_text(frame, "SYSTEM MONITOR", (20, 40), 0.7, (255, 255, 255))
    risk_color = (0, 0, 255) if risk_score > 50 else (0, 255, 0)
    draw_hud_text(frame, f"Risk Level: {risk_score}/100", (20, 80), 0.6, risk_color)

    if results.multi_face_landmarks:
        no_faces_frames = 0 
        
        if len(results.multi_face_landmarks) > 1:
            multi_faces_frames += 1
            if multi_faces_frames > 30:
                risk_score = min(100, risk_score + 50)
                multi_faces_frames = 0
            draw_hud_text(frame, "CRITICAL: MULTIPLE FACES!", (20, 240), 0.6, (0, 0, 255))
            
        else:
            multi_faces_frames = 0
            for face_landmarks in results.multi_face_landmarks:
                
                # EAR Calculation
                left_ear, _ = calculate_ear(LEFT_EYE, face_landmarks, w, h)
                right_ear, _ = calculate_ear(RIGHT_EYE, face_landmarks, w, h)
                avg_ear = (left_ear + right_ear) / 2.0
                
                if avg_ear < 0.20:
                    eyes_closed_frames += 1
                    if eyes_closed_frames > 60:
                        risk_score = min(100, risk_score + 10)
                        eyes_closed_frames = 0
                else:
                    eyes_closed_frames = 0
                
                # solvePnP Calculation
                image_points = np.array([(int(face_landmarks.landmark[idx].x * w), int(face_landmarks.landmark[idx].y * h)) for idx in TRACKED_FACE_INDICES], dtype=np.float32)
                _, rotation_vector, translation_vector = cv2.solvePnP(model_points, image_points, camera_matrix, dist_coeffs, flags=cv2.SOLVEPNP_ITERATIVE)
                rotation_matrix, _ = cv2.Rodrigues(rotation_vector)
                proj_matrix = np.hstack((rotation_matrix, translation_vector))
                _, _, _, _, _, _, euler_angles = cv2.decomposeProjectionMatrix(proj_matrix)
                
                pitch = euler_angles[0][0]
                yaw = euler_angles[1][0]
                
                # Print Telemetry
                draw_hud_text(frame, f"EAR: {avg_ear:.2f}", (20, 130), 0.5, (200, 200, 200))
                draw_hud_text(frame, f"Yaw: {int(yaw)} deg", (20, 160), 0.5, (200, 200, 200))
                draw_hud_text(frame, f"Pitch: {int(pitch)} deg", (20, 190), 0.5, (200, 200, 200))

                # Jitter-Proof Thresholds
                if abs(yaw) > 35 or pitch < -170:
                    head_turn_frames += 1
                    draw_hud_text(frame, "WARN: HEAD ANOMALY", (20, 240), 0.6, (0, 165, 255))
                    if head_turn_frames > 60:
                        risk_score = min(100, risk_score + 20)
                        head_turn_frames = 0
                else:
                    # Decay logic to stop millisecond flicker
                    head_turn_frames = max(0, head_turn_frames - 2)
                    if eyes_closed_frames > 15:
                        draw_hud_text(frame, "WARN: EYES CLOSED", (20, 240), 0.6, (0, 165, 255))
                    else:
                        draw_hud_text(frame, "STATUS: NOMINAL", (20, 240), 0.6, (0, 255, 0))

    else:
        no_faces_frames += 1
        if no_faces_frames > 30:
            risk_score = min(100, risk_score + 25)
            no_faces_frames = 0
        draw_hud_text(frame, "STATUS: NO FACE DETECTED", (20, 240), 0.6, (0, 0, 255))

    cv2.imshow('Ultimate AI Proctoring System', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

Loading Models...
